# Model 3b: SARIMAX Model + Exogenous Features for Drug `N02BA`

## Hyperparameter Selection Methodology:
Exogenous calendar regressors (`is_weekend`, `sin_dayofyear`, `cos_dayofyear`) evaluated alongside seasonal ARMA components.


In [1]:
# Dynamic Dependency Guard & Environment Initialization
import sys, subprocess, os

def install_and_import(pkg, module_name=None):
    if module_name is None:
        module_name = pkg
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import('numpy')
install_and_import('pandas')
install_and_import('matplotlib')
install_and_import('seaborn')
install_and_import('scikit-learn', 'sklearn')
install_and_import('statsmodels')
install_and_import('lightgbm')
install_and_import('xgboost')
install_and_import('shap')
install_and_import('prophet')
install_and_import('optuna')
install_and_import('torch')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.sans-serif': 'Inter, Roboto, Arial, sans-serif',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--'
})

TARGET_DRUG = 'N02BA'
data_dir = r'c:\Users\ranje\sales forcasting\times_series\dataset'

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df   = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series   = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series  = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series     = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    rmse  = np.sqrt(np.mean((y_true - y_pred)**2))
    mae   = np.mean(np.abs(y_true - y_pred))
    wape  = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE (%)': wape}

print(f"Dataset for {TARGET_DRUG} loaded successfully!")
print(f"  * Train  : {train_series.index.min().strftime('%Y-%m-%d')} to {train_series.index.max().strftime('%Y-%m-%d')} ({len(train_series)} days)")
print(f"  * Val    : {val_series.index.min().strftime('%Y-%m-%d')} to {val_series.index.max().strftime('%Y-%m-%d')} ({len(val_series)} days)")
print(f"  * Test   : {test_series.index.min().strftime('%Y-%m-%d')} to {test_series.index.max().strftime('%Y-%m-%d')} ({len(test_series)} days)")


Dataset for N02BA loaded successfully!
  * Train  : 2014-01-02 to 2017-12-31 (1460 days)
  * Val    : 2018-01-01 to 2018-12-31 (365 days)
  * Test   : 2019-01-01 to 2019-10-08 (281 days)


In [2]:
# Step 1: Exogenous Regressor Preparation & Fit
def get_exog(df_index):
    dof = df_index.dayofyear
    dow = df_index.dayofweek
    exog = pd.DataFrame(index=df_index)
    exog['is_weekend'] = (dow >= 5).astype(float)
    exog['sin_dayofyear'] = np.sin(2 * np.pi * dof / 365.25)
    exog['cos_dayofyear'] = np.cos(2 * np.pi * dof / 365.25)
    return exog

exog_tr = get_exog(train_series.index)
exog_va = get_exog(val_series.index)
exog_cb = get_exog(combined_series.index)
exog_ts = get_exog(test_series.index)

from statsmodels.tsa.statespace.sarimax import SARIMAX

m3b_val_model = SARIMAX(np.log1p(train_series), exog=exog_tr, order=(1,1,1), seasonal_order=(1,1,1,7), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
pred_val_log = m3b_val_model.forecast(steps=len(val_series), exog=exog_va)
val_rmsle = evaluate_metrics(val_series.values, np.clip(np.expm1(pred_val_log), 0, None))['RMSLE']
print(f"Validation Set RMSLE for SARIMAX + Exog: {val_rmsle:.6f}")

m3b_model = SARIMAX(np.log1p(combined_series), exog=exog_cb, order=(1,1,1), seasonal_order=(1,1,1,7), enforce_stationarity=False, enforce_invertibility=False)
m3b_fit = m3b_model.fit(disp=False)

pred_log = m3b_fit.forecast(steps=len(test_series), exog=exog_ts)
m3b_test_pred = np.clip(np.expm1(pred_log), 0, None)
m3b_test_pred.index = test_series.index

test_metrics = evaluate_metrics(test_series, m3b_test_pred)
print(f"=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 3b: SARIMAX + EXOG ===")
for k, v in test_metrics.items():
    print(f"  * {k:10s}: {v:.4f}")

pd.DataFrame({'date': test_series.index, 'pred_SARIMAX': m3b_test_pred.values}).to_csv('m3_sarimax_preds.csv', index=False)


Validation Set RMSLE for SARIMAX + Exog: 0.562373


=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 3b: SARIMAX + EXOG ===
  * RMSLE     : 0.5651
  * RMSE      : 2.2491
  * MAE       : 1.7024
  * WAPE (%)  : 54.3743
